# Módulo 5 · Notebook 2 — Agente ReAct

ReAct = **Reason + Act**.

El agente alterna entre:
1. Razonar qué necesita.
2. Llamar una tool.
3. Observar el resultado.
4. Responder al usuario.

En este notebook construimos ese ciclo con `create_react_agent`.

## 1. Setup

In [1]:
import warnings
from statistics import mean

from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langgraph.prebuilt import create_react_agent

warnings.filterwarnings("ignore")

WORLD_CUP_CHAMPIONS = {
    1930: "Uruguay", 1934: "Italia", 1938: "Italia", 1950: "Uruguay",
    1954: "Alemania Federal", 1958: "Brasil", 1962: "Brasil", 1966: "Inglaterra",
    1970: "Brasil", 1974: "Alemania Federal", 1978: "Argentina", 1982: "Italia",
    1986: "Argentina", 1990: "Alemania", 1994: "Brasil", 1998: "Francia",
    2002: "Brasil", 2006: "Italia", 2010: "España",
}
GOALS_PER_WORLD_CUP = {
    1930: 70, 1934: 70, 1938: 84, 1950: 88, 1954: 140, 1958: 126,
    1962: 89, 1966: 89, 1970: 95, 1974: 97, 1978: 102, 1982: 146,
    1986: 132, 1990: 115, 1994: 141, 1998: 171, 2002: 161, 2006: 147, 2010: 145,
}

llm = ChatOllama(model="llama3.2:3b", temperature=0)
print("✅ Setup listo")

/Users/guane/Documentos/GlogalLogic/AI-course/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


✅ Setup listo


In [2]:
@tool
def campeon_del_mundial(anio: int) -> str:
    """Devuelve el campeón del Mundial para un año específico."""
    return WORLD_CUP_CHAMPIONS.get(anio, f"No tengo datos del año {anio}.")


@tool
def promedio_goles(inicio: int, fin: int) -> str:
    """Calcula el promedio de goles entre dos mundiales (años incluidos)."""
    years = [y for y in GOALS_PER_WORLD_CUP if inicio <= y <= fin]
    if not years:
        return "No hay mundiales en ese rango."
    avg = mean(GOALS_PER_WORLD_CUP[y] for y in years)
    return f"Promedio de goles entre {inicio} y {fin}: {avg:.2f}"


tools = [campeon_del_mundial, promedio_goles]
print("Tools:", [t.name for t in tools])

Tools: ['campeon_del_mundial', 'promedio_goles']


## 2. Crear agente ReAct

In [3]:
SYSTEM_PROMPT = """
Eres un asistente experto en historia de los mundiales de fútbol.
- Si necesitas datos exactos, usa una tool.
- Si la pregunta se puede responder sin tools, responde directo.
- Responde siempre en español y de forma breve.
"""

agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=SYSTEM_PROMPT,
)

print("✅ Agente ReAct creado")

✅ Agente ReAct creado


## 3. Ejecutar una consulta y revisar trazas

In [4]:
query = "¿Quién ganó el mundial de 1998 y cuál fue el promedio de goles entre 1990 y 2010?"

result = agent.invoke({"messages": [("user", query)]})
messages = result["messages"]

print(f"Total de mensajes en la traza: {len(messages)}\n")
for m in messages:
    role = m.type
    content = m.content if isinstance(m.content, str) else str(m.content)
    print(f"[{role}] {content[:220]}\n")

Total de mensajes en la traza: 5

[human] ¿Quién ganó el mundial de 1998 y cuál fue el promedio de goles entre 1990 y 2010?

[ai] 

[tool] Francia

[tool] Promedio de goles entre 1990 y 2010: 146.67

[ai] Para calcular el promedio de goles, utilicé la herramienta para obtener los datos del mundial de 1998 y luego calculé el promedio de goles entre 1990 y 2010.

En cuanto al mundial de 1998, la Francia ganó el título.

Par



In [5]:
final_answer = messages[-1].content
print("Respuesta final del agente:\n")
print(final_answer)

Respuesta final del agente:

Para calcular el promedio de goles, utilicé la herramienta para obtener los datos del mundial de 1998 y luego calculé el promedio de goles entre 1990 y 2010.

En cuanto al mundial de 1998, la Francia ganó el título.

Para calcular el promedio de goles, utilicé los siguientes pasos:

1. Obtuve la cantidad total de goles anotados en cada mundial desde 1990 hasta 2010.
2. Sumé todos los goles anotados en ese período.
3. Dividí la suma por el número de años (21) para obtener el promedio.

El resultado es un promedio de goles de 146,67 por partido.


## 4. Resumen

- ReAct combina razonamiento y uso de herramientas de forma iterativa.
- `create_react_agent` automatiza el loop de tool-calling.
- La salida incluye toda la traza en `messages`, útil para depurar.

Siguiente: un patrón **multiagente** (router + especialistas).